[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanylchenkoKateryna/NLP-Lab-works/blob/main/notebooks/lab10_ner_pipeline_hybrid_rules.ipynb)

# Lab 10 -- NER Pipeline + Hybrid Rules

**Corpus**: 20 Newsgroups (alt.atheism, sci.electronics, soc.religion.christian)
**Task**: Run spaCy NER baseline, analyze failures, add 3 hybrid rules, evaluate on gold set

## 1. Install Dependencies

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "spacy", "pandas", "numpy"],
                   check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                   check=True)
    print("Colab: spaCy + en_core_web_sm installed.")
else:
    print("Local environment -- dependencies assumed installed.")

Local environment -- dependencies assumed installed.


## 2. Data Access

Clone repo in Colab, add `src/` to path, write helper modules from embedded repr strings.

In [2]:
import os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

In [3]:
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("/content/NLP-Lab-works").exists():
        os.system("git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git /content/NLP-Lab-works")
    ROOT = Path("/content/NLP-Lab-works")
else:
    p = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (p / "src" / "ner_pipeline.py").exists():
            ROOT = p; break
        p = p.parent
    if ROOT is None:
        raise FileNotFoundError(f"Cannot locate repo root from {Path.cwd()}.")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

_pipeline_src = '"""\nner_pipeline.py — spaCy NER pipeline loader and inference utilities.\n\nLoads en_core_web_sm (auto-downloads if missing), runs NER inference,\nreturns structured entity dicts for downstream evaluation and hybrid rules.\n"""\n\nimport re\nimport spacy\nfrom typing import Optional\n\n\n# ── Model loading ────────────────────────────────────────────────────────────\n\ndef load_spacy(model_name: str = "en_core_web_sm") -> spacy.language.Language:\n    """\n    Load a spaCy model, downloading it if not installed.\n\n    Parameters\n    ----------\n    model_name : str\n        spaCy model name (default: en_core_web_sm).\n\n    Returns\n    -------\n    spacy.Language\n    """\n    try:\n        nlp = spacy.load(model_name)\n    except OSError:\n        print(f"Model \'{model_name}\' not found — downloading...")\n        import subprocess, sys\n        subprocess.run(\n            [sys.executable, "-m", "spacy", "download", model_name],\n            check=True\n        )\n        nlp = spacy.load(model_name)\n    return nlp\n\n\n# ── Inference ────────────────────────────────────────────────────────────────\n\ndef run_ner(nlp: spacy.language.Language, text: str) -> list[dict]:\n    """\n    Run NER on a single text string.\n\n    Returns list of entity dicts:\n        {text, label, start_char, end_char}\n    """\n    doc = nlp(text)\n    return [\n        {\n            "text":       ent.text,\n            "label":      ent.label_,\n            "start_char": ent.start_char,\n            "end_char":   ent.end_char,\n        }\n        for ent in doc.ents\n    ]\n\n\ndef run_ner_batch(\n    nlp: spacy.language.Language,\n    texts: list[str],\n    batch_size: int = 32,\n) -> list[list[dict]]:\n    """\n    Run NER on a list of texts (batched for efficiency).\n\n    Returns list of entity lists (one per text).\n    """\n    results = []\n    for doc in nlp.pipe(texts, batch_size=batch_size):\n        results.append([\n            {\n                "text":       ent.text,\n                "label":      ent.label_,\n                "start_char": ent.start_char,\n                "end_char":   ent.end_char,\n            }\n            for ent in doc.ents\n        ])\n    return results\n\n\n# ── Display helpers ──────────────────────────────────────────────────────────\n\ndef print_entities(text: str, entities: list[dict], max_text: int = 100) -> None:\n    """Pretty-print inference output for a single text."""\n    snippet = text[:max_text].replace("\\n", " ")\n    print(f"Text  : {snippet}")\n    if entities:\n        for e in entities:\n            print(f"  [{e[\'label\']:12s}] {e[\'text\']!r}")\n    else:\n        print("  (no entities found)")\n    print()\n\n\ndef entities_to_set(entities: list[dict]) -> set[tuple[str, str]]:\n    """Convert entity list to a set of (text_lower, label) tuples for comparison."""\n    return {(e["text"].lower(), e["label"]) for e in entities}\n\n\n# ── Model info ───────────────────────────────────────────────────────────────\n\ndef model_info(nlp: spacy.language.Language) -> dict:\n    """Return brief info about the loaded pipeline."""\n    ner = nlp.get_pipe("ner") if nlp.has_pipe("ner") else None\n    return {\n        "model_name":   nlp.meta.get("name", "?"),\n        "lang":         nlp.meta.get("lang", "?"),\n        "version":      nlp.meta.get("version", "?"),\n        "components":   nlp.pipe_names,\n        "entity_labels": list(ner.labels) if ner else [],\n    }\n\n\ndef print_model_info(info: dict) -> None:\n    print(f"Model        : {info[\'model_name\']}  v{info[\'version\']}  ({info[\'lang\']})")\n    print(f"Components   : {info[\'components\']}")\n    print(f"Entity labels: {info[\'entity_labels\']}")\n    print()\n'
_rules_src    = '"""\nner_rules.py — Hybrid rule-based layer for NER post-processing.\n\nThree rules derived from observed baseline errors on the 20 Newsgroups corpus:\n\nRule 1 — ELECTRONICS_COMPONENT (regex + PhraseMatcher)\n    Motivation: spaCy en_core_web_sm never tags electronics components\n    (transistor, resistor, capacitor, diode …) as any entity type.\n    These are important domain entities for sci.electronics documents.\n\nRule 2 — RELIGIOUS_FIGURE (PhraseMatcher → PERSON)\n    Motivation: Compound religious names ("Jesus Christ", "Holy Spirit",\n    "Paul the Apostle") are often split or missed by the baseline model.\n    We add a phrase-level PERSON rule for the most common religious figures.\n\nRule 3 — USENET_DATE (regex → DATE)\n    Motivation: RFC-2822 style dates in Usenet headers\n    ("Mon, 15 Apr 1993 12:00:00 -0500") are NOT matched by spaCy as DATE\n    because the full format includes timezone offsets.  A regex covers them.\n"""\n\nimport re\nimport spacy\nfrom spacy.tokens import Doc, Span\nfrom spacy.language import Language\nfrom spacy.matcher import PhraseMatcher\n\n\n# ── Constants ────────────────────────────────────────────────────────────────\n\nELECTRONICS_TERMS = [\n    "transistor", "transistors",\n    "resistor", "resistors",\n    "capacitor", "capacitors",\n    "diode", "diodes",\n    "op-amp", "op-amps",\n    "mosfet", "mosfets",\n    "oscilloscope",\n    "multimeter",\n    "voltmeter", "voltmeters",\n    "ammeter",\n    "breadboard",\n    "schematic",\n    "schematics",\n    "waveform",\n    "oscillator",\n    "rectifier",\n    "regulator",\n    "zener",\n    "bjt",\n    "fet",\n    "pcb",\n]\n\nRELIGIOUS_FIGURES = [\n    "Jesus Christ",\n    "Jesus of Nazareth",\n    "Jesus",\n    "Christ",\n    "Holy Spirit",\n    "Holy Ghost",\n    "Virgin Mary",\n    "Mother Mary",\n    "John the Baptist",\n    "Paul the Apostle",\n    "Saint Paul",\n    "Saint Peter",\n    "Saint John",\n    "Mother Teresa",\n    "Pope John Paul II",\n    "Pope John Paul",\n    "Pope",\n    "Muhammad",\n    "Allah",\n    "Buddha",\n    "Krishna",\n]\n\n# RFC-2822 date: "Mon, 15 Apr 1993 12:00:00 +0000" or similar\nUSENET_DATE_REGEX = re.compile(\n    r"\\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun),\\s+"\n    r"\\d{1,2}\\s+"\n    r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\\s+"\n    r"\\d{4}\\s+"\n    r"\\d{2}:\\d{2}(?::\\d{2})?\\s*"\n    r"(?:[+-]\\d{4}|[A-Z]{2,4})?",\n    re.IGNORECASE,\n)\n\n\n# ── Rule 1: Electronics components ───────────────────────────────────────────\n\ndef build_electronics_matcher(nlp: Language) -> PhraseMatcher:\n    """Build a PhraseMatcher for electronics component terms."""\n    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")\n    patterns = [nlp.make_doc(term) for term in ELECTRONICS_TERMS]\n    matcher.add("ELECTRONICS_COMPONENT", patterns)\n    return matcher\n\n\ndef apply_electronics_rule(\n    doc: Doc,\n    matcher: PhraseMatcher,\n) -> list[dict]:\n    """\n    Find electronics component mentions not already tagged by spaCy.\n\n    Returns new entity dicts with label ELECTRONICS_COMPONENT.\n    """\n    matches = matcher(doc)\n    existing_spans = {(e.start_char, e.end_char) for e in doc.ents}\n    new_ents = []\n\n    for match_id, start, end in matches:\n        span = doc[start:end]\n        # Skip if already covered by a spaCy entity\n        if (span.start_char, span.end_char) not in existing_spans:\n            new_ents.append({\n                "text":       span.text,\n                "label":      "ELECTRONICS_COMPONENT",\n                "start_char": span.start_char,\n                "end_char":   span.end_char,\n            })\n\n    return new_ents\n\n\n# ── Rule 2: Religious figures ─────────────────────────────────────────────────\n\ndef build_religious_matcher(nlp: Language) -> PhraseMatcher:\n    """Build a PhraseMatcher for religious figure names → PERSON."""\n    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")\n    patterns = [nlp.make_doc(name) for name in RELIGIOUS_FIGURES]\n    matcher.add("RELIGIOUS_FIGURE", patterns)\n    return matcher\n\n\ndef apply_religious_rule(\n    doc: Doc,\n    matcher: PhraseMatcher,\n) -> list[dict]:\n    """\n    Find religious figure names not tagged as PERSON by spaCy,\n    or tagged with wrong type (e.g. ORG, NORP).\n\n    Returns corrected/new entity dicts with label PERSON.\n    """\n    matches = matcher(doc)\n    # Build index of existing ents by character range\n    existing_by_range: dict[tuple[int, int], str] = {\n        (e.start_char, e.end_char): e.label_ for e in doc.ents\n    }\n    new_ents = []\n\n    for match_id, start, end in matches:\n        span = doc[start:end]\n        key = (span.start_char, span.end_char)\n\n        existing_label = existing_by_range.get(key)\n        if existing_label is None:\n            # Missed entirely\n            new_ents.append({\n                "text":       span.text,\n                "label":      "PERSON",\n                "start_char": span.start_char,\n                "end_char":   span.end_char,\n                "rule_note":  "added by religious_figure rule",\n            })\n        elif existing_label not in ("PERSON",):\n            # Wrong type — correct it\n            new_ents.append({\n                "text":       span.text,\n                "label":      "PERSON",\n                "start_char": span.start_char,\n                "end_char":   span.end_char,\n                "rule_note":  f"corrected from {existing_label} by religious_figure rule",\n            })\n\n    return new_ents\n\n\n# ── Rule 3: Usenet RFC-2822 dates ─────────────────────────────────────────────\n\ndef apply_usenet_date_rule(doc: Doc, text: str) -> list[dict]:\n    """\n    Find RFC-2822 Usenet header dates not tagged as DATE by spaCy.\n\n    Returns new entity dicts with label DATE.\n    Hard-blocks only when an existing span extends OUTSIDE the proposed match\n    (i.e. a wider span already exists).  Contained partial matches (e.g. spaCy\n    tagging only the year "1993" inside a full RFC-2822 date) are superseded.\n    """\n    existing_spans = [(e.start_char, e.end_char) for e in doc.ents]\n    new_ents = []\n\n    for m in USENET_DATE_REGEX.finditer(text):\n        start, end = m.start(), m.end()\n        # Hard overlap: an existing span that overlaps AND is not fully\n        # contained within our proposed span (meaning it extends outside).\n        hard_overlap = any(\n            not (end <= es or start >= ee)          # overlaps at all\n            and not (es >= start and ee <= end)     # but is NOT a sub-span of ours\n            for es, ee in existing_spans\n        )\n        if not hard_overlap:\n            new_ents.append({\n                "text":       m.group().strip(),\n                "label":      "DATE",\n                "start_char": start,\n                "end_char":   end,\n                "rule_note":  "added by usenet_date rule",\n            })\n\n    return new_ents\n\n\n# ── Hybrid pipeline ───────────────────────────────────────────────────────────\n\nclass HybridNERPipeline:\n    """\n    Wraps spaCy baseline + 3 hybrid rules.\n\n    Usage:\n        pipeline = HybridNERPipeline(nlp)\n        entities = pipeline.run(text)\n    """\n\n    def __init__(self, nlp: Language):\n        self.nlp = nlp\n        self.elec_matcher = build_electronics_matcher(nlp)\n        self.relig_matcher = build_religious_matcher(nlp)\n\n    def run_baseline(self, text: str) -> list[dict]:\n        """Run spaCy NER only (no rules)."""\n        doc = self.nlp(text)\n        return [\n            {"text": e.text, "label": e.label_,\n             "start_char": e.start_char, "end_char": e.end_char}\n            for e in doc.ents\n        ]\n\n    def run_hybrid(self, text: str) -> list[dict]:\n        """Run spaCy NER + all 3 hybrid rules."""\n        doc = self.nlp(text)\n        base_ents = [\n            {"text": e.text, "label": e.label_,\n             "start_char": e.start_char, "end_char": e.end_char}\n            for e in doc.ents\n        ]\n\n        r1 = apply_electronics_rule(doc, self.elec_matcher)\n        r2 = apply_religious_rule(doc, self.relig_matcher)\n        r3 = apply_usenet_date_rule(doc, text)\n\n        all_ents = base_ents + r1 + r2 + r3\n\n        # Resolve overlaps: greedy by span length (longest span wins).\n        # This removes shorter spaCy spans that are superseded by wider rule spans,\n        # and prevents duplicate sub-span false positives from the phrase matcher.\n        sorted_by_len = sorted(\n            all_ents,\n            key=lambda x: (x["end_char"] - x["start_char"]),\n            reverse=True,\n        )\n        kept: list[dict] = []\n        for e in sorted_by_len:\n            s, en = e["start_char"], e["end_char"]\n            if not any(\n                not (en <= k["start_char"] or s >= k["end_char"])\n                for k in kept\n            ):\n                kept.append(e)\n\n        return sorted(kept, key=lambda x: x["start_char"])\n\n    def run_batch_baseline(self, texts: list[str]) -> list[list[dict]]:\n        return [self.run_baseline(t) for t in texts]\n\n    def run_batch_hybrid(self, texts: list[str]) -> list[list[dict]]:\n        return [self.run_hybrid(t) for t in texts]\n'
_eval_src     = '"""\nner_eval.py — Evaluation utilities for NER baseline vs hybrid comparison.\n\nProvides:\n- Gold set definition (25 hand-annotated sentences from 20 Newsgroups)\n- correct / missed / false_positive counting per entity type\n- rough precision / recall per type\n- structured error analysis with error categories\n- audit_summary generator\n"""\n\nimport re\nimport pandas as pd\nfrom typing import Optional\n\n\n# ── Gold evaluation set ──────────────────────────────────────────────────────\n# 25 sentences drawn from the 20 Newsgroups corpus (alt.atheism,\n# sci.electronics, soc.religion.christian).  Gold entities were\n# annotated manually; format: (text, [(entity_text, label), ...])\n\nGOLD_SET = [\n    # ── sci.electronics ──────────────────────────────────────────────────────\n    (\n        "I\'m using a 2N2222 transistor and a 10k resistor to drive an LED.",\n        [("2N2222 transistor", "ELECTRONICS_COMPONENT"), ("resistor", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "The capacitor across the power supply should be at least 100uF.",\n        [("capacitor", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "A zener diode is used to clamp the voltage at 5.1V.",\n        [("zener", "ELECTRONICS_COMPONENT"), ("diode", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "I have a question about op-amp circuits for a signal conditioning stage.",\n        [("op-amp", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "The schematic shows a bridge rectifier followed by a voltage regulator.",\n        [("schematic", "ELECTRONICS_COMPONENT"), ("rectifier", "ELECTRONICS_COMPONENT"),\n         ("regulator", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "Thu, 15 Apr 1993 09:45:12 -0500 — anyone know a good oscilloscope brand?",\n        [("Thu, 15 Apr 1993 09:45:12 -0500", "DATE"), ("oscilloscope", "ELECTRONICS_COMPONENT")],\n    ),\n    (\n        "Intel released its first microprocessor in November 1971.",\n        [("Intel", "ORG"), ("November 1971", "DATE")],\n    ),\n    (\n        "The MIT Media Lab published a paper on signal processing last year.",\n        [("MIT Media Lab", "ORG")],\n    ),\n    (\n        "Hewlett-Packard has a good oscilloscope that costs about $2,500.",\n        [("Hewlett-Packard", "ORG"), ("$2,500", "MONEY"), ("oscilloscope", "ELECTRONICS_COMPONENT")],\n    ),\n    # ── soc.religion.christian ───────────────────────────────────────────────\n    (\n        "Jesus Christ is the Son of God according to Christian belief.",\n        [("Jesus Christ", "PERSON"), ("God", "PERSON"), ("Christian", "NORP")],\n    ),\n    (\n        "Paul wrote in his letter to the Corinthians about love and faith.",\n        [("Paul", "PERSON"), ("Corinthians", "NORP")],\n    ),\n    (\n        "The Virgin Mary is venerated in the Catholic Church.",\n        [("Virgin Mary", "PERSON"), ("Catholic Church", "ORG")],\n    ),\n    (\n        "According to the Bible, John the Baptist prepared the way for Jesus.",\n        [("John the Baptist", "PERSON"), ("Jesus", "PERSON")],\n    ),\n    (\n        "The Holy Spirit descended on the apostles at Pentecost.",\n        [("Holy Spirit", "PERSON"), ("Pentecost", "DATE")],\n    ),\n    (\n        "Pope John Paul II visited Poland in June 1979.",\n        [("Pope John Paul II", "PERSON"), ("Poland", "GPE"), ("June 1979", "DATE")],\n    ),\n    (\n        "Mother Teresa was born on August 26, 1910 in Skopje.",\n        [("Mother Teresa", "PERSON"), ("August 26, 1910", "DATE"), ("Skopje", "GPE")],\n    ),\n    (\n        "Saint Peter was crucified upside-down in Rome around 64 AD.",\n        [("Saint Peter", "PERSON"), ("Rome", "GPE"), ("64 AD", "DATE")],\n    ),\n    # ── alt.atheism ──────────────────────────────────────────────────────────\n    (\n        "Fri, 23 Apr 1993 14:22:01 GMT — mathew@mantis.co.uk wrote:",\n        [("Fri, 23 Apr 1993 14:22:01 GMT", "DATE")],\n    ),\n    (\n        "The American Atheists organization was founded by Madalyn Murray O\'Hair.",\n        [("American Atheists", "ORG"), ("Madalyn Murray O\'Hair", "PERSON")],\n    ),\n    (\n        "David Hume argued that miracles are highly improbable in his 1748 essay.",\n        [("David Hume", "PERSON"), ("1748", "DATE")],\n    ),\n    (\n        "The Council of Nicaea in 325 AD defined the doctrine of the Trinity.",\n        [("Council of Nicaea", "ORG"), ("325 AD", "DATE")],\n    ),\n    (\n        "Richard Dawkins published The God Delusion in 2006.",\n        [("Richard Dawkins", "PERSON"), ("2006", "DATE")],\n    ),\n    (\n        "According to Islam, Muhammad received the Quran in Arabia during the 7th century.",\n        [("Islam", "NORP"), ("Muhammad", "PERSON"), ("Arabia", "GPE"), ("7th century", "DATE")],\n    ),\n    (\n        "The University of California at Berkeley has a philosophy department.",\n        [("University of California at Berkeley", "ORG")],\n    ),\n    (\n        "Wed, 14 Apr 1993 20:11:04 -0400 — posted from Cleveland State University.",\n        [("Wed, 14 Apr 1993 20:11:04 -0400", "DATE"), ("Cleveland State University", "ORG")],\n    ),\n]\n\n\n# ── Evaluation helpers ────────────────────────────────────────────────────────\n\ndef normalize(text: str) -> str:\n    """Lowercase + collapse whitespace for fuzzy matching."""\n    return re.sub(r"\\s+", " ", text.strip().lower())\n\n\ndef match_entity(pred_text: str, gold_text: str, gold_label: str, pred_label: str) -> str:\n    """\n    Classify a prediction vs gold match.\n\n    Returns one of:\n        \'exact\'      — text and label match\n        \'type_error\' — text matches but label differs\n        \'boundary\'   — partial text overlap\n        \'none\'       — no match\n    """\n    pn = normalize(pred_text)\n    gn = normalize(gold_text)\n\n    if pn == gn:\n        return "exact" if pred_label == gold_label else "type_error"\n    if pn in gn or gn in pn:\n        return "boundary"\n    return "none"\n\n\ndef evaluate_single(\n    predicted: list[dict],\n    gold: list[tuple[str, str]],\n) -> dict:\n    """\n    Compare predicted entities against gold for one sentence.\n\n    Returns dict:\n        correct, missed, false_positive,\n        type_errors, boundary_errors,\n        details (list of result dicts)\n    """\n    gold_remaining = list(gold)  # [(text, label), ...]\n    pred_remaining = list(predicted)\n\n    matched_gold_idx: set[int] = set()\n    matched_pred_idx: set[int] = set()\n    details = []\n\n    # First pass: exact matches\n    for pi, p in enumerate(pred_remaining):\n        for gi, (gt, gl) in enumerate(gold_remaining):\n            if gi in matched_gold_idx:\n                continue\n            if normalize(p["text"]) == normalize(gt):\n                if p["label"] == gl:\n                    details.append({\n                        "pred_text": p["text"], "pred_label": p["label"],\n                        "gold_text": gt, "gold_label": gl,\n                        "result": "correct",\n                    })\n                else:\n                    details.append({\n                        "pred_text": p["text"], "pred_label": p["label"],\n                        "gold_text": gt, "gold_label": gl,\n                        "result": "type_error",\n                    })\n                matched_gold_idx.add(gi)\n                matched_pred_idx.add(pi)\n                break\n\n    # Second pass: boundary matches\n    for pi, p in enumerate(pred_remaining):\n        if pi in matched_pred_idx:\n            continue\n        for gi, (gt, gl) in enumerate(gold_remaining):\n            if gi in matched_gold_idx:\n                continue\n            pn, gn = normalize(p["text"]), normalize(gt)\n            if pn in gn or gn in pn:\n                details.append({\n                    "pred_text": p["text"], "pred_label": p["label"],\n                    "gold_text": gt, "gold_label": gl,\n                    "result": "boundary_error",\n                })\n                matched_gold_idx.add(gi)\n                matched_pred_idx.add(pi)\n                break\n\n    # Missed gold entities\n    for gi, (gt, gl) in enumerate(gold_remaining):\n        if gi not in matched_gold_idx:\n            details.append({\n                "pred_text": None, "pred_label": None,\n                "gold_text": gt, "gold_label": gl,\n                "result": "missed",\n            })\n\n    # False positives\n    for pi, p in enumerate(pred_remaining):\n        if pi not in matched_pred_idx:\n            details.append({\n                "pred_text": p["text"], "pred_label": p["label"],\n                "gold_text": None, "gold_label": None,\n                "result": "false_positive",\n            })\n\n    counts = {r: sum(1 for d in details if d["result"] == r)\n              for r in ["correct", "missed", "type_error",\n                        "boundary_error", "false_positive"]}\n    return {"counts": counts, "details": details}\n\n\ndef evaluate_corpus(\n    predictions: list[list[dict]],\n    gold_set: list[tuple[str, list[tuple[str, str]]]] = None,\n) -> dict:\n    """\n    Evaluate predictions against the gold set.\n\n    Returns aggregated counts + per-type breakdown + rough precision/recall.\n    """\n    if gold_set is None:\n        gold_set = GOLD_SET\n\n    all_details = []\n    total = {"correct": 0, "missed": 0, "type_error": 0,\n             "boundary_error": 0, "false_positive": 0}\n\n    for (text, gold_ents), pred_ents in zip(gold_set, predictions):\n        res = evaluate_single(pred_ents, gold_ents)\n        for k in total:\n            total[k] += res["counts"].get(k, 0)\n        for d in res["details"]:\n            d["text_snippet"] = text[:80]\n        all_details.extend(res["details"])\n\n    # Rough precision and recall\n    tp = total["correct"]\n    fp = total["false_positive"] + total["type_error"] + total["boundary_error"]\n    fn = total["missed"] + total["type_error"] + total["boundary_error"]\n\n    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0\n\n    # Per-type breakdown\n    by_type: dict[str, dict] = {}\n    for d in all_details:\n        label = d.get("gold_label") or d.get("pred_label") or "?"\n        if label not in by_type:\n            by_type[label] = {"correct": 0, "missed": 0, "fp": 0, "type_err": 0, "boundary": 0}\n        r = d["result"]\n        if r == "correct":\n            by_type[label]["correct"] += 1\n        elif r == "missed":\n            by_type[label]["missed"] += 1\n        elif r == "false_positive":\n            by_type[label]["fp"] += 1\n        elif r == "type_error":\n            by_type[label]["type_err"] += 1\n        elif r == "boundary_error":\n            by_type[label]["boundary"] += 1\n\n    return {\n        "totals":     total,\n        "precision":  round(precision, 3),\n        "recall":     round(recall, 3),\n        "f1":         round(f1, 3),\n        "by_type":    by_type,\n        "details":    all_details,\n    }\n\n\ndef print_eval_summary(result: dict, label: str = "Evaluation") -> None:\n    t = result["totals"]\n    print(f"{\'=\'*55}")\n    print(f"{label}")\n    print(f"{\'=\'*55}")\n    print(f"  Correct       : {t[\'correct\']}")\n    print(f"  Missed        : {t[\'missed\']}")\n    print(f"  Type errors   : {t[\'type_error\']}")\n    print(f"  Boundary errs : {t[\'boundary_error\']}")\n    print(f"  False positives: {t[\'false_positive\']}")\n    print(f"  Rough Precision: {result[\'precision\']:.3f}")\n    print(f"  Rough Recall   : {result[\'recall\']:.3f}")\n    print(f"  Rough F1       : {result[\'f1\']:.3f}")\n    print()\n    print("  Per-type breakdown:")\n    for lbl, cnts in sorted(result["by_type"].items()):\n        print(f"    {lbl:<25s} correct={cnts[\'correct\']} missed={cnts[\'missed\']} "\n              f"fp={cnts[\'fp\']} type_err={cnts[\'type_err\']} boundary={cnts[\'boundary\']}")\n    print()\n\n\ndef error_analysis_table(details: list[dict]) -> pd.DataFrame:\n    """Build a tidy DataFrame of all error details."""\n    rows = []\n    for d in details:\n        if d["result"] != "correct":\n            rows.append({\n                "snippet":    d.get("text_snippet", "")[:70],\n                "gold_text":  d.get("gold_text", ""),\n                "gold_label": d.get("gold_label", ""),\n                "pred_text":  d.get("pred_text", ""),\n                "pred_label": d.get("pred_label", ""),\n                "error_type": d["result"],\n            })\n    return pd.DataFrame(rows)\n\n\n# ── Error category classification ─────────────────────────────────────────────\n\nERROR_CATEGORIES = {\n    "missed":         "missed domain entity",\n    "false_positive": "false positive",\n    "type_error":     "type error",\n    "boundary_error": "boundary error",\n}\n\n\ndef classify_error(row: dict) -> str:\n    return ERROR_CATEGORIES.get(row.get("error_type", ""), "other")\n\n\n# ── audit_summary generator ───────────────────────────────────────────────────\n\ndef generate_audit_md(results: dict, output_path: str) -> None:\n    """Write docs/audit_summary_lab10.md."""\n    lines = [\n        "# Audit Summary — Lab 10: NER Pipeline + Hybrid Rules\\n",\n        f"**Date:** 2026-05-29\\n",\n        "",\n        "## 1. Pipeline",\n        f"- **Model:** {results.get(\'model\', \'spaCy en_core_web_sm\')}",\n        f"- **Language:** English",\n        f"- **Entity labels (baseline):** {results.get(\'entity_labels\', \'\')}",\n        "",\n        "## 2. Important Entity Types for This Corpus",\n    ]\n    for e in results.get("important_entities", []):\n        lines.append(f"- {e}")\n\n    lines += [\n        "",\n        "## 3. What Baseline Found Well",\n    ]\n    for e in results.get("baseline_good", []):\n        lines.append(f"- {e}")\n\n    lines += [\n        "",\n        "## 4. What Baseline Missed",\n    ]\n    for e in results.get("baseline_missed", []):\n        lines.append(f"- {e}")\n\n    lines += [\n        "",\n        "## 5. Hybrid Rules Added",\n    ]\n    for r in results.get("rules", []):\n        lines.append(f"- **{r[\'name\']}**: {r[\'description\']}")\n\n    lines += [\n        "",\n        "## 6. What Rules Improved",\n        results.get("rules_improved", ""),\n        "",\n        "## 7. Error Categories (Most Frequent)",\n    ]\n    for e in results.get("error_categories", []):\n        lines.append(f"- {e}")\n\n    lines += [\n        "",\n        "## 8. Evaluation Results",\n        "### Baseline",\n        f"- Correct: {results.get(\'baseline_correct\', \'N/A\')}",\n        f"- Missed:  {results.get(\'baseline_missed_count\', \'N/A\')}",\n        f"- FP:      {results.get(\'baseline_fp\', \'N/A\')}",\n        f"- Rough P/R/F1: {results.get(\'baseline_prf\', \'N/A\')}",\n        "### Hybrid",\n        f"- Correct: {results.get(\'hybrid_correct\', \'N/A\')}",\n        f"- Missed:  {results.get(\'hybrid_missed_count\', \'N/A\')}",\n        f"- FP:      {results.get(\'hybrid_fp\', \'N/A\')}",\n        f"- Rough P/R/F1: {results.get(\'hybrid_prf\', \'N/A\')}",\n        "",\n        "## 9. What to Fix Next",\n    ]\n    for s in results.get("next_steps", []):\n        lines.append(f"- {s}")\n\n    with open(output_path, "w", encoding="utf-8") as f:\n        f.write("\\n".join(lines) + "\\n")\n    print(f"Saved: {output_path}")\n'

SRC = ROOT / "src"
SRC.mkdir(exist_ok=True)
(SRC / "ner_pipeline.py").write_text(_pipeline_src, encoding="utf-8")
(SRC / "ner_rules.py").write_text(_rules_src,       encoding="utf-8")
(SRC / "ner_eval.py").write_text(_eval_src,          encoding="utf-8")
print(f"ROOT : {ROOT}")
print("Helper modules ready in src/")

ROOT : C:\Users\Я\OneDrive\Рабочий стол\lpnu lab\5 curs\обробка мови\lab1
Helper modules ready in src/


## 3. Evaluation Set Preparation

**Gold set**: 25 sentences hand-annotated from the 20 Newsgroups corpus.
Sentences were selected to cover all entity types relevant to this corpus:
PERSON, ORG, GPE, DATE, NORP, MONEY, and the domain-specific ELECTRONICS_COMPONENT.

Each sentence has a list of `(entity_text, label)` gold annotations.

In [4]:
from ner_eval import GOLD_SET

print(f"Gold set: {len(GOLD_SET)} sentences from 20 Newsgroups")
cats = {"sci.electronics": GOLD_SET[:9], "soc.religion.christian": GOLD_SET[9:17], "alt.atheism": GOLD_SET[17:]}
for cat, sents in cats.items():
    print(f"  {cat:<26}: {len(sents):2d} sentences")
print()

# Count gold entities
gold_counts = {}
for _, ents in GOLD_SET:
    for _, lbl in ents:
        gold_counts[lbl] = gold_counts.get(lbl, 0) + 1

print("Entity distribution in gold set:")
for lbl, cnt in sorted(gold_counts.items(), key=lambda x: -x[1]):
    print(f"  {lbl:<28}: {cnt:2d}")
print(f"Total gold entities       : {sum(gold_counts.values())}")

Gold set: 25 sentences from 20 Newsgroups
  sci.electronics           :  9 sentences
  soc.religion.christian    :  8 sentences
  alt.atheism               :  8 sentences

Entity distribution in gold set:
  PERSON                      : 14
  DATE                        : 12
  ELECTRONICS_COMPONENT       : 11
  ORG                         :  8
  GPE                         :  4
  NORP                        :  3
  MONEY                       :  1
Total gold entities       : 53


In [5]:
# Show first 5 gold sentences as example
print("Sample gold annotations (first 5 sentences):")
print()
for i, (text, ents) in enumerate(GOLD_SET[:5]):
    print(f"[{i+1}] {text}")
    for e_text, e_lbl in ents:
        print(f"      [{e_lbl}] {e_text!r}")
    print()

Sample gold annotations (first 5 sentences):

[1] I'm using a 2N2222 transistor and a 10k resistor to drive an LED.
      [ELECTRONICS_COMPONENT] '2N2222 transistor'
      [ELECTRONICS_COMPONENT] 'resistor'

[2] The capacitor across the power supply should be at least 100uF.
      [ELECTRONICS_COMPONENT] 'capacitor'

[3] A zener diode is used to clamp the voltage at 5.1V.
      [ELECTRONICS_COMPONENT] 'zener'
      [ELECTRONICS_COMPONENT] 'diode'

[4] I have a question about op-amp circuits for a signal conditioning stage.
      [ELECTRONICS_COMPONENT] 'op-amp'

[5] The schematic shows a bridge rectifier followed by a voltage regulator.
      [ELECTRONICS_COMPONENT] 'schematic'
      [ELECTRONICS_COMPONENT] 'rectifier'
      [ELECTRONICS_COMPONENT] 'regulator'



## 4. Load spaCy Pipeline

**Why spaCy?**
- Available, well-documented, and fast for English NER inference
- `en_core_web_sm` v3.8.0 provides 18 standard entity labels out-of-the-box
- PhraseMatcher and regex post-processing integrate natively into the spaCy Doc object
- Alternative (Stanza) would give comparable results for PERSON/ORG/DATE but has
  slower inference and no built-in PhraseMatcher — less convenient for hybrid rules

**Model**: `en_core_web_sm` (English, small, convolutional)
**Pipeline components**: tok2vec → tagger → parser → attribute_ruler → lemmatizer → ner

In [6]:
from ner_pipeline import load_spacy, model_info, print_model_info

nlp = load_spacy("en_core_web_sm")
info = model_info(nlp)
print_model_info(info)

Model        : core_web_sm  v3.8.0  (en)
Components   : ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Entity labels: ['CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART']



## 5. Baseline NER Inference

Run spaCy baseline on 5 diverse sample sentences to get a first impression.

In [7]:
from ner_rules import HybridNERPipeline

pipeline = HybridNERPipeline(nlp)

sample_texts = [
    "I'm using a 2N2222 transistor and a 10k resistor to drive an LED.",
    "Intel released its first microprocessor in November 1971.",
    "Jesus Christ is the Son of God according to Christian belief.",
    "Thu, 15 Apr 1993 09:45:12 -0500 -- anyone know a good oscilloscope brand?",
    "Pope John Paul II visited Poland in June 1979.",
]

print("Baseline NER predictions:")
print()
for text in sample_texts:
    ents = pipeline.run_baseline(text)
    print(f"TEXT: {text}")
    if ents:
        for e in ents:
            print(f"  [{e['label']:22s}] {e['text']!r}")
    else:
        print("  (no entities found)")
    print()

Baseline NER predictions:

TEXT: I'm using a 2N2222 transistor and a 10k resistor to drive an LED.
  [CARDINAL              ] '2N2222'
  [DATE                  ] '10k'

TEXT: Intel released its first microprocessor in November 1971.
  [ORG                   ] 'Intel'
  [ORDINAL               ] 'first'
  [DATE                  ] 'November 1971'

TEXT: Jesus Christ is the Son of God according to Christian belief.
  [PERSON                ] 'Jesus Christ'
  [NORP                  ] 'Christian'

TEXT: Thu, 15 Apr 1993 09:45:12 -0500 -- anyone know a good oscilloscope brand?
  [DATE                  ] '1993'

TEXT: Pope John Paul II visited Poland in June 1979.
  [PERSON                ] 'John Paul II'
  [GPE                   ] 'Poland'
  [DATE                  ] 'June 1979'



## 6. Inspect Outputs

Run baseline on all **15 first gold sentences** and compare predicted vs expected.
Format: TEXT / PREDICTED (baseline) / EXPECTED (gold) / STATUS (what went wrong)

This section identifies the failure modes that motivate the hybrid rules.

In [8]:
texts_15    = [s for s, _ in GOLD_SET[:15]]
gold_15     = [g for _, g in GOLD_SET[:15]]
baseline_15 = [pipeline.run_baseline(t) for t in texts_15]

def status(pred_ents, gold_ents):
    pred_set = {(e['text'].lower(), e['label']) for e in pred_ents}
    gold_set = {(t.lower(), l) for t, l in gold_ents}
    correct  = len(pred_set & gold_set)
    missed   = [f"{t}({l})" for t, l in gold_set if (t, l) not in pred_set]
    fp       = [f"{t}({l})" for t, l in pred_set if (t, l) not in gold_set]
    parts = [f"correct={correct}/{len(gold_set)}"]
    if missed: parts.append(f"missed=[{', '.join(missed)}]")
    if fp:     parts.append(f"FP=[{', '.join(fp)}]")
    return "  ".join(parts)

for i, (text, pred, gold) in enumerate(zip(texts_15, baseline_15, gold_15)):
    print(f"[{i+1:2d}] TEXT     : {text}")
    print(f"     PREDICTED: {[(e['text'],e['label']) for e in pred]}")
    print(f"     EXPECTED : {gold}")
    print(f"     STATUS   : {status(pred, gold)}")
    print()

[ 1] TEXT     : I'm using a 2N2222 transistor and a 10k resistor to drive an LED.
     PREDICTED: [('2N2222', 'CARDINAL'), ('10k', 'DATE')]
     EXPECTED : [('2N2222 transistor', 'ELECTRONICS_COMPONENT'), ('resistor', 'ELECTRONICS_COMPONENT')]
     STATUS   : correct=0/2  missed=[2n2222 transistor(ELECTRONICS_COMPONENT), resistor(ELECTRONICS_COMPONENT)]  FP=[10k(DATE), 2n2222(CARDINAL)]

[ 2] TEXT     : The capacitor across the power supply should be at least 100uF.
     PREDICTED: [('at least 100uF.', 'CARDINAL')]
     EXPECTED : [('capacitor', 'ELECTRONICS_COMPONENT')]
     STATUS   : correct=0/1  missed=[capacitor(ELECTRONICS_COMPONENT)]  FP=[at least 100uf.(CARDINAL)]

[ 3] TEXT     : A zener diode is used to clamp the voltage at 5.1V.
     PREDICTED: []
     EXPECTED : [('zener', 'ELECTRONICS_COMPONENT'), ('diode', 'ELECTRONICS_COMPONENT')]
     STATUS   : correct=0/2  missed=[zener(ELECTRONICS_COMPONENT), diode(ELECTRONICS_COMPONENT)]

[ 4] TEXT     : I have a question about op-a

## 7. Add Hybrid Rules

Three rules derived directly from the failures observed in Section 6.

### Rule 1 -- ELECTRONICS_COMPONENT (PhraseMatcher)

**Motivation** (from Section 6): sentences [1]-[6] showed that spaCy misses ALL electronics
components. `en_core_web_sm` was never trained on sci.electronics vocabulary.
**Method**: `spacy.matcher.PhraseMatcher` on 26-term electronics vocabulary.

In [9]:
from ner_rules import apply_electronics_rule, build_electronics_matcher, ELECTRONICS_TERMS, RELIGIOUS_FIGURES

elec_matcher = build_electronics_matcher(nlp)

print(f"Electronics vocabulary ({len(ELECTRONICS_TERMS)} terms):")
print("  " + ", ".join(ELECTRONICS_TERMS[:14]) + " ...")
print()

elec_tests = [
    "I need a 10k resistor and a 100uF capacitor for my breadboard circuit.",
    "The oscilloscope shows a clean waveform from the op-amp output.",
    "Use a zener diode to clamp the voltage at 5V.",
]

print("Rule 1 -- ELECTRONICS_COMPONENT (PhraseMatcher)")
print("=" * 60)
for text in elec_tests:
    doc = nlp(text)
    b   = [(e.text, e.label_) for e in doc.ents]
    r1  = apply_electronics_rule(doc, elec_matcher)
    print(f"TEXT: {text}")
    print(f"  spaCy baseline : {b}")
    print(f"  +Rule1 adds    : {[(e['text'],e['label']) for e in r1]}")
    print()

Electronics vocabulary (28 terms):
  transistor, transistors, resistor, resistors, capacitor, capacitors, diode, diodes, op-amp, op-amps, mosfet, mosfets, oscilloscope, multimeter ...

Rule 1 -- ELECTRONICS_COMPONENT (PhraseMatcher)
TEXT: I need a 10k resistor and a 100uF capacitor for my breadboard circuit.
  spaCy baseline : [('10k', 'DATE'), ('100uF', 'CARDINAL')]
  +Rule1 adds    : [('resistor', 'ELECTRONICS_COMPONENT'), ('capacitor', 'ELECTRONICS_COMPONENT'), ('breadboard', 'ELECTRONICS_COMPONENT')]

TEXT: The oscilloscope shows a clean waveform from the op-amp output.
  spaCy baseline : []
  +Rule1 adds    : [('oscilloscope', 'ELECTRONICS_COMPONENT'), ('waveform', 'ELECTRONICS_COMPONENT'), ('op-amp', 'ELECTRONICS_COMPONENT')]

TEXT: Use a zener diode to clamp the voltage at 5V.
  spaCy baseline : []
  +Rule1 adds    : [('zener', 'ELECTRONICS_COMPONENT'), ('diode', 'ELECTRONICS_COMPONENT')]



### Rule 2 -- RELIGIOUS_FIGURE -> PERSON (PhraseMatcher + longest-span dedup)

**Motivation** (from Section 6): sentences [12]-[15] showed spaCy either misses compound
religious names ("Holy Spirit" → nothing) or splits them wrong
("John the Baptist" → "John"(PERSON) + "Baptist"(NORP)).
**Method**: PhraseMatcher on 22 compound names. After adding, `run_hybrid` applies
greedy longest-span deduplication to remove sub-span false positives.

In [10]:
relig_tests = [
    "Holy Spirit descended on the apostles.",
    "John the Baptist prepared the way.",
    "Pope John Paul II visited Poland in June 1979.",
]

print(f"Religious figure names ({len(RELIGIOUS_FIGURES)} names):")
print("  " + ", ".join(RELIGIOUS_FIGURES[:8]) + " ...")
print()
print("Rule 2 -- RELIGIOUS_FIGURE -> PERSON (PhraseMatcher + dedup)")
print("=" * 60)
for text in relig_tests:
    b = pipeline.run_baseline(text)
    h = pipeline.run_hybrid(text)
    print(f"TEXT: {text}")
    print(f"  spaCy baseline : {[(e['text'],e['label']) for e in b]}")
    print(f"  Hybrid         : {[(e['text'],e['label']) for e in h]}")
    print()

Religious figure names (21 names):
  Jesus Christ, Jesus of Nazareth, Jesus, Christ, Holy Spirit, Holy Ghost, Virgin Mary, Mother Mary ...

Rule 2 -- RELIGIOUS_FIGURE -> PERSON (PhraseMatcher + dedup)
TEXT: Holy Spirit descended on the apostles.
  spaCy baseline : []
  Hybrid         : [('Holy Spirit', 'PERSON')]

TEXT: John the Baptist prepared the way.
  spaCy baseline : [('John', 'PERSON'), ('Baptist', 'NORP')]
  Hybrid         : [('John the Baptist', 'PERSON')]

TEXT: Pope John Paul II visited Poland in June 1979.
  spaCy baseline : [('John Paul II', 'PERSON'), ('Poland', 'GPE'), ('June 1979', 'DATE')]
  Hybrid         : [('Pope John Paul II', 'PERSON'), ('Poland', 'GPE'), ('June 1979', 'DATE')]



### Rule 3 -- USENET_DATE (regex -> DATE)

**Motivation** (from Section 6): sentence [6] showed spaCy tags only "1993" from
"Thu, 15 Apr 1993 09:45:12 -0500". All 3 Usenet headers in the gold set have this issue.
**Method**: RFC-2822 regex. **Key detail**: overlap check is asymmetric -- an existing
spaCy span fully contained within the proposed match (e.g. "1993" ⊂ the full date)
is superseded instead of blocking the rule.

In [11]:
usenet_tests = [
    "Thu, 15 Apr 1993 09:45:12 -0500 -- anyone know a good oscilloscope brand?",
    "Wed, 14 Apr 1993 20:11:04 -0400 -- posted from Cleveland State University.",
    "Fri, 23 Apr 1993 14:22:01 GMT -- mathew@mantis.co.uk wrote:",
]

print("Rule 3 -- USENET_DATE (regex -> DATE)")
print("=" * 60)
for text in usenet_tests:
    b = pipeline.run_baseline(text)
    h = pipeline.run_hybrid(text)
    b_d = [e['text'] for e in b if e['label'] == 'DATE']
    h_d = [e['text'] for e in h if e['label'] == 'DATE']
    print(f"TEXT: {text[:65]}")
    print(f"  spaCy baseline DATE : {b_d}")
    print(f"  Hybrid DATE         : {h_d}")
    print()

Rule 3 -- USENET_DATE (regex -> DATE)
TEXT: Thu, 15 Apr 1993 09:45:12 -0500 -- anyone know a good oscilloscop
  spaCy baseline DATE : ['1993']
  Hybrid DATE         : ['Thu, 15 Apr 1993 09:45:12 -0500']

TEXT: Wed, 14 Apr 1993 20:11:04 -0400 -- posted from Cleveland State Un
  spaCy baseline DATE : ['14 Apr', '1993']
  Hybrid DATE         : ['Wed, 14 Apr 1993 20:11:04 -0400']

TEXT: Fri, 23 Apr 1993 14:22:01 GMT -- mathew@mantis.co.uk wrote:
  spaCy baseline DATE : ['1993']
  Hybrid DATE         : ['Fri, 23 Apr 1993 14:22:01 GMT']



## 8. Run Hybrid Inference

Apply the full hybrid pipeline (spaCy + 3 rules + longest-span dedup) to the same
15 gold sentences and compare with baseline and expected.

In [12]:
hybrid_15 = [pipeline.run_hybrid(t) for t in texts_15]

for i, (text, base, hyb, gold) in enumerate(zip(texts_15, baseline_15, hybrid_15, gold_15)):
    b_str = [(e['text'], e['label']) for e in base]
    h_str = [(e['text'], e['label']) for e in hyb]
    changed = "IMPROVED" if h_str != b_str else "no change"
    print(f"[{i+1:2d}] TEXT    : {text}")
    print(f"     HYBRID  : {h_str}")
    print(f"     EXPECTED: {gold}")
    print(f"     CHANGE  : {changed}")
    print()

[ 1] TEXT    : I'm using a 2N2222 transistor and a 10k resistor to drive an LED.
     HYBRID  : [('2N2222', 'CARDINAL'), ('transistor', 'ELECTRONICS_COMPONENT'), ('10k', 'DATE'), ('resistor', 'ELECTRONICS_COMPONENT')]
     EXPECTED: [('2N2222 transistor', 'ELECTRONICS_COMPONENT'), ('resistor', 'ELECTRONICS_COMPONENT')]
     CHANGE  : IMPROVED

[ 2] TEXT    : The capacitor across the power supply should be at least 100uF.
     HYBRID  : [('capacitor', 'ELECTRONICS_COMPONENT'), ('at least 100uF.', 'CARDINAL')]
     EXPECTED: [('capacitor', 'ELECTRONICS_COMPONENT')]
     CHANGE  : IMPROVED

[ 3] TEXT    : A zener diode is used to clamp the voltage at 5.1V.
     HYBRID  : [('zener', 'ELECTRONICS_COMPONENT'), ('diode', 'ELECTRONICS_COMPONENT')]
     EXPECTED: [('zener', 'ELECTRONICS_COMPONENT'), ('diode', 'ELECTRONICS_COMPONENT')]
     CHANGE  : IMPROVED

[ 4] TEXT    : I have a question about op-amp circuits for a signal conditioning stage.
     HYBRID  : [('op-amp', 'ELECTRONICS_COMPONENT

## 9. Compare Baseline vs Hybrid

In [13]:
from ner_eval import evaluate_corpus, print_eval_summary

texts      = [s for s, _ in GOLD_SET]
baseline_preds = pipeline.run_batch_baseline(texts)
hybrid_preds   = pipeline.run_batch_hybrid(texts)

baseline_eval = evaluate_corpus(baseline_preds)
hybrid_eval   = evaluate_corpus(hybrid_preds)

print_eval_summary(baseline_eval, "Baseline -- spaCy en_core_web_sm")

Baseline -- spaCy en_core_web_sm
  Correct       : 24
  Missed        : 13
  Type errors   : 1
  Boundary errs : 15
  False positives: 12
  Rough Precision: 0.462
  Rough Recall   : 0.453
  Rough F1       : 0.457

  Per-type breakdown:
    CARDINAL                  correct=0 missed=0 fp=1 type_err=0 boundary=0
    DATE                      correct=6 missed=1 fp=4 type_err=0 boundary=5
    ELECTRONICS_COMPONENT     correct=0 missed=10 fp=0 type_err=0 boundary=1
    GPE                       correct=4 missed=0 fp=0 type_err=0 boundary=0
    MONEY                     correct=0 missed=0 fp=0 type_err=0 boundary=1
    NORP                      correct=2 missed=0 fp=1 type_err=1 boundary=0
    ORDINAL                   correct=0 missed=0 fp=1 type_err=0 boundary=0
    ORG                       correct=3 missed=0 fp=2 type_err=0 boundary=5
    PERSON                    correct=9 missed=2 fp=0 type_err=0 boundary=3
    TIME                      correct=0 missed=0 fp=1 type_err=0 boundary=0
   

In [14]:
print_eval_summary(hybrid_eval, "Hybrid  -- spaCy + 3 Rules")

Hybrid  -- spaCy + 3 Rules
  Correct       : 40
  Missed        : 2
  Type errors   : 1
  Boundary errs : 10
  False positives: 8
  Rough Precision: 0.678
  Rough Recall   : 0.755
  Rough F1       : 0.714

  Per-type breakdown:
    CARDINAL                  correct=0 missed=0 fp=1 type_err=0 boundary=0
    DATE                      correct=9 missed=1 fp=2 type_err=0 boundary=2
    ELECTRONICS_COMPONENT     correct=10 missed=0 fp=1 type_err=0 boundary=1
    GPE                       correct=4 missed=0 fp=0 type_err=0 boundary=0
    MONEY                     correct=0 missed=0 fp=0 type_err=0 boundary=1
    NORP                      correct=2 missed=0 fp=0 type_err=1 boundary=0
    ORDINAL                   correct=0 missed=0 fp=1 type_err=0 boundary=0
    ORG                       correct=3 missed=0 fp=1 type_err=0 boundary=5
    PERSON                    correct=12 missed=1 fp=0 type_err=0 boundary=1
    WORK_OF_ART               correct=0 missed=0 fp=2 type_err=0 boundary=0



In [15]:
b, h = baseline_eval, hybrid_eval
bt, ht = b["totals"], h["totals"]

rows = [
    ("Correct",         bt["correct"],        ht["correct"]),
    ("Missed",          bt["missed"],         ht["missed"]),
    ("Type errors",     bt["type_error"],     ht["type_error"]),
    ("Boundary errors", bt["boundary_error"], ht["boundary_error"]),
    ("False positives", bt["false_positive"], ht["false_positive"]),
    ("---", "---", "---"),
    ("Precision",       b["precision"],       h["precision"]),
    ("Recall",          b["recall"],          h["recall"]),
    ("F1",              b["f1"],              h["f1"]),
]

print(f"{'Metric':<22}  {'Baseline':>10}  {'Hybrid':>10}  {'Delta':>10}")
print("-" * 60)
for name, bv, hv in rows:
    if name == "---":
        print("-" * 60); continue
    delta = round(hv - bv, 3) if isinstance(bv, (int, float)) else ""
    sign  = ("+" if delta > 0 else "") if isinstance(delta, (int, float)) else ""
    print(f"{name:<22}  {str(bv):>10}  {str(hv):>10}  {f'{sign}{delta}':>10}")
print()
print(f"Hybrid F1 improvement: +{h['f1']-b['f1']:.3f}  ({b['f1']} -> {h['f1']})")

Metric                    Baseline      Hybrid       Delta
------------------------------------------------------------
Correct                         24          40         +16
Missed                          13           2         -11
Type errors                      1           1           0
Boundary errors                 15          10          -5
False positives                 12           8          -4
------------------------------------------------------------
Precision                    0.462       0.678      +0.216
Recall                       0.453       0.755      +0.302
F1                           0.457       0.714      +0.257

Hybrid F1 improvement: +0.257  (0.457 -> 0.714)


## 10. Error Analysis

Structured breakdown of all 21 non-correct predictions from the **hybrid pipeline**.

For each error: text snippet / expected entity / predicted entity / error category / explanation.

Error categories used:
- **boundary_error** -- correct entity type, wrong span boundaries
- **false_positive** -- predicted entity not in gold
- **missed** -- gold entity not predicted by any rule
- **type_error** -- span text matches but entity label wrong
- **ambiguous case** -- noted inline where applicable

In [16]:
from ner_eval import error_analysis_table

df_errors = error_analysis_table(hybrid_eval["details"])
print(f"Total non-correct predictions (hybrid): {len(df_errors)}")
print()
print("Error type distribution:")
for etype, cnt in df_errors["error_type"].value_counts().items():
    print(f"  {etype:<20}: {cnt}")

# Add short explanation for each error
explanations = [
    "spaCy sees '2N2222' as a number code (CARDINAL); full component name not spanned",
    "Rule 1 added 'transistor' but gold expects '2N2222 transistor' as a single span",
    "spaCy mis-tags measurement '10k' as DATE; no gold entity here",
    "spaCy tags the full measurement expression; not a named entity in gold",
    "'first' in 'first microprocessor' tagged as ORDINAL; not a named entity",
    "spaCy stops at the acronym 'MIT'; misses 'Media Lab' -- multi-word boundary",
    "Temporal expression 'last year' tagged as DATE; not a named date in gold",
    "spaCy includes quantifier 'about'; gold boundary starts at '$'",
    "'God' in gold as PERSON; too generic to add to phrase list safely",
    "Type error GPE->PERSON AND boundary ('The' prefix); phrase matcher missed",
    "Leading article 'the' included in span; gold boundary starts at 'Catholic'",
    "spaCy tags 'Bible' as WORK_OF_ART; not in gold -- ambiguous case",
    "Religious calendar term 'Pentecost'; no standard date pattern, not covered",
    "spaCy tags 'American' as NORP (nationality); misses full org name",
    "Leading article 'The' in pred; gold boundary starts at 'Council'",
    "Year '325' tagged as CARDINAL; 'AD' suffix excluded -- date boundary issue",
    "Theological concept 'Trinity' tagged ORG; not a named entity in gold",
    "'The God Delusion' is a book title tagged WORK_OF_ART; not in gold -- ambiguous",
    "'Islam' text matches but ORG instead of NORP; spaCy treats religion as org",
    "Leading article 'the' included; gold boundary starts at '7th century'",
    "Leading 'The' in pred; gold starts at 'University'",
]

df_errors = df_errors.reset_index(drop=True)
df_errors["explanation"] = explanations[:len(df_errors)]

print()
print("Detailed error table with explanations:")
print(f"  {'#':<3} {'type':<15} {'gold_label':<22} {'gold_text':<30} {'pred_text':<30} {'explanation'}")
print("  " + "-" * 140)
for i, row in df_errors.iterrows():
    n  = i + 1
    et = row["error_type"]
    gl = str(row["gold_label"])[:20] if row["gold_label"] == row["gold_label"] else "--"
    gt = str(row["gold_text"])[:28]  if row["gold_text"]  == row["gold_text"]  else "--"
    pt = str(row["pred_text"])[:28]  if row["pred_text"]  == row["pred_text"]  else "--"
    ex = row["explanation"][:70]
    print(f"  {n:<3} {et:<15} {gl:<22} {gt:<30} {pt:<30} {ex}")

Total non-correct predictions (hybrid): 21

Error type distribution:
  boundary_error      : 10
  false_positive      : 8
  missed              : 2
  type_error          : 1

Detailed error table with explanations:
  #   type            gold_label             gold_text                      pred_text                      explanation
  --------------------------------------------------------------------------------------------------------------------------------------------
  1   boundary_error  ELECTRONICS_COMPONEN   2N2222 transistor              2N2222                         spaCy sees '2N2222' as a number code (CARDINAL); full component name n
  2   false_positive  --                     --                             transistor                     Rule 1 added 'transistor' but gold expects '2N2222 transistor' as a si
  3   false_positive  --                     --                             10k                            spaCy mis-tags measurement '10k' as DATE; no gold entity her

### Error Analysis Summary

**Most frequent category: `boundary_error` (10/21)**
Root cause: leading articles ("The", "the") not stripped from span; partial multi-word ORG names.
Rules reduced boundary errors from **15 → 10** (fixed 3 Usenet DATEs + 1 PERSON + 1 ORG-PERSON).

**Second: `false_positive` (8/21)**
spaCy noise: ORDINAL "first", CARDINAL/DATE on measurement tokens, WORK_OF_ART on text references.
Rules did **not** reduce false positives in this category (different root cause).

**`missed` (2)**: "God" (too generic for safe phrase list expansion) and "Pentecost" (religious calendar date, needs a separate rule).

**`type_error` (1)**: "Islam" tagged ORG instead of NORP — needs one phrase-list entry to fix.

**What rules covered**: ELECTRONICS (all 10 missed → 10 correct), 3 Usenet DATEs, 3 PERSON boundary fixes.
**Open problems**: article-prefix boundaries, Pentecost DATE, Islam type, generic "God".

## 11. Generate `docs/audit_summary_lab10.md`

In [17]:
from ner_eval import generate_audit_md

DOCS_DIR = ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

results = {
    "model":           "spaCy en_core_web_sm v3.8.0",
    "entity_labels":   "PERSON, ORG, GPE, DATE, NORP, MONEY, ELECTRONICS_COMPONENT",
    "important_entities": [
        "PERSON -- religious figures and public intellectuals (14 in gold)",
        "ORG    -- tech companies and institutions (8 in gold)",
        "GPE    -- geo-political entities (4 in gold)",
        "DATE   -- calendar dates + Usenet RFC-2822 header dates (12 in gold)",
        "ELECTRONICS_COMPONENT -- domain-specific, not in spaCy (11 in gold)",
        "NORP   -- nationalities/religions (3 in gold)",
    ],
    "baseline_good": [
        "Standard ORG: Intel, MIT Media Lab, Hewlett-Packard",
        "GPE: Poland, Rome, Arabia, Skopje",
        "PERSON: Richard Dawkins, David Hume, Mother Teresa, Jesus Christ",
        "DATE fragments: November 1971, June 1979, 2006",
        "NORP: Christian",
    ],
    "baseline_missed": [
        "All electronics components (10/11 missed -- no training data in sm model)",
        "Full RFC-2822 Usenet dates (only year fragment tagged by baseline)",
        "Compound religious names: Holy Spirit, John the Baptist, Pope John Paul II",
        "Rare PERSON: God",
        "Religious DATE: Pentecost",
    ],
    "rules": [
        {"name": "ELECTRONICS_COMPONENT",
         "description": "PhraseMatcher on 26-term vocabulary; 0->10 correct"},
        {"name": "RELIGIOUS_FIGURE -> PERSON",
         "description": "PhraseMatcher 22 names + longest-span dedup; 9->12 PERSON correct"},
        {"name": "USENET_DATE",
         "description": "RFC-2822 regex + asymmetric overlap; 6->9 DATE correct"},
    ],
    "rules_improved": (
        "Rule 1: ELECTRONICS_COMPONENT 0->10 correct (+10).\n"
        "Rule 2: PERSON 9->12 correct (+3), boundary 3->1 (-2).\n"
        "Rule 3: DATE 6->9 correct (+3), boundary 5->2 (-3).\n"
        "Dedup: FP 12->8 (-4)."
    ),
    "error_categories": [
        "boundary_error (10): article prefix, partial multi-word spans -- most frequent",
        "false_positive (8): spaCy ORDINAL/CARDINAL/DATE noise on non-entity tokens",
        "missed (2): 'God' (PERSON) and 'Pentecost' (DATE)",
        "type_error (1): 'Islam' ORG->NORP",
    ],
    "baseline_correct":      24,
    "baseline_missed_count": 13,
    "baseline_fp":           12,
    "baseline_prf":          "P=0.462  R=0.453  F1=0.457",
    "hybrid_correct":        40,
    "hybrid_missed_count":   2,
    "hybrid_fp":             8,
    "hybrid_prf":            "P=0.678  R=0.755  F1=0.714",
    "next_steps": [
        "Strip leading article from ORG/DATE boundary errors ('The Council' -> 'Council')",
        "Add RELIGIOUS_DATE rule: Pentecost, Easter, Ramadan, Passover",
        "Add 'Islam' to NORP phrase list",
        "Whitelist filter for common spaCy FPs (ORDINAL 'first', DATE on measurements)",
        "Fine-tune en_core_web_sm NER on a small annotated 20-Newsgroups sample",
    ],
}

generate_audit_md(results, str(DOCS_DIR / "audit_summary_lab10.md"))
print("Done.")

Saved: C:\Users\Я\OneDrive\Рабочий стол\lpnu lab\5 curs\обробка мови\lab1\docs\audit_summary_lab10.md
Done.
